# Competition research with embeddings via OpenRouter

This notebook is designed to be replicated by workshop students. It calls OpenRouter's OpenAI-compatible embeddings endpoint, classifies a small competition-policy corpus from a human-labelled gold set, and visualizes the result.

**Important:** use a test key with a small spending limit. Do not upload confidential or restricted research material.

In [ ]:
!pip -q install openai pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os, getpass, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from google.colab import files
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

print('Upload competition-corpus.csv from the workshop materials.')
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
display(df.head())
print(df.groupby(['split','category']).size())

## 1. Connect to OpenRouter

Create a key at [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys), set a small limit, and paste it only into the private Colab prompt below. The key is kept in memory for this runtime and is not written into the notebook.

In [ ]:
OPENROUTER_API_KEY = getpass.getpass('Paste your OpenRouter API key (input is hidden): ')
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=OPENROUTER_API_KEY)
EMBEDDING_MODEL = 'openai/text-embedding-3-small'
print('Client ready:', EMBEDDING_MODEL)

## 2. Prepare the gold set and held-out test set

The gold set is the small set labelled by researchers. The test set is held out so we can measure performance rather than simply admire the output.

In [ ]:
gold = df[df.split == 'gold'].copy().reset_index(drop=True)
test = df[df.split == 'test'].copy().reset_index(drop=True)
categories = sorted(gold.category.unique())
print('Gold:', len(gold), 'Test:', len(test))
print('Categories:', categories)

## 3. Request embeddings in batches

An embedding is a vector representation of text. OpenRouter exposes an embeddings endpoint that follows the familiar OpenAI client pattern. The batch function below makes the request reproducible and keeps the code visible to students.

In [ ]:
def embed_texts(texts, batch_size=16, pause_seconds=0.2):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        ordered = sorted(response.data, key=lambda item: item.index)
        vectors.extend([item.embedding for item in ordered])
        print(f'Embedded {min(start+batch_size, len(texts))}/{len(texts)}')
        time.sleep(pause_seconds)
    return np.asarray(vectors, dtype='float32')

gold_vectors = embed_texts(gold.text.tolist())
test_vectors = embed_texts(test.text.tolist())
print(gold_vectors.shape, test_vectors.shape)

## 4. Classify by nearest category centroid

For each category, calculate the average embedding of its gold examples. A test paragraph receives the category whose centroid has the highest cosine similarity. This is intentionally simple: students can see the relationship between labels, vectors, and predictions.

In [ ]:
centroids = {}
for category in categories:
    centroids[category] = gold_vectors[gold.category.to_numpy() == category].mean(axis=0)
centroid_matrix = np.vstack([centroids[c] for c in categories])
centroid_matrix = centroid_matrix / np.linalg.norm(centroid_matrix, axis=1, keepdims=True)

scores = test_vectors @ centroid_matrix.T
predicted_index = scores.argmax(axis=1)
test['prediction'] = np.array(categories)[predicted_index]
test['similarity'] = scores.max(axis=1)

print('Accuracy:', round(accuracy_score(test.category, test.prediction), 3))
print(classification_report(test.category, test.prediction, zero_division=0))

## 5. Visualize the embedding space

PCA compresses the vectors to two dimensions for a visual overview. The plot is exploratory: a visible cluster is not proof of a valid category or causal relationship.

In [ ]:
all_vectors = np.vstack([gold_vectors, test_vectors])
plot_vectors = PCA(n_components=2, random_state=42).fit_transform(all_vectors)
plot_df = pd.concat([gold.assign(kind='gold'), test.assign(kind='test')], ignore_index=True)
plot_df['x'] = plot_vectors[:,0]
plot_df['y'] = plot_vectors[:,1]

plt.figure(figsize=(12,8))
sns.scatterplot(data=plot_df, x='x', y='y', hue='category', style='kind', s=130, palette='tab10')
for _, row in plot_df.iterrows():
    plt.text(row.x+0.03, row.y+0.03, str(row.id), fontsize=9)
plt.title('Competition corpus in embedding space (PCA)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 6. Inspect uncertainty and nearest examples

Low similarity is a routing signal, not a verdict. In a real workflow, send these rows to a researcher for review and preserve the decision in an audit table.

In [ ]:
THRESHOLD = 0.55
review_queue = test[test.similarity < THRESHOLD].sort_values('similarity')
print(f'{len(review_queue)} of {len(test)} test rows below threshold {THRESHOLD}')
display(review_queue[['id','text','category','prediction','similarity','source_url']])

nearest_gold = cosine_similarity(test_vectors, gold_vectors).argmax(axis=1)
test['nearest_gold_id'] = gold.iloc[nearest_gold].id.to_numpy()
test['nearest_gold_category'] = gold.iloc[nearest_gold].category.to_numpy()
display(test[['id','category','prediction','similarity','nearest_gold_id','nearest_gold_category']])

In [ ]:
cm = confusion_matrix(test.category, test.prediction, labels=categories)
plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='crest', xticklabels=categories, yticklabels=categories)
plt.xlabel('Predicted category'); plt.ylabel('True category'); plt.xticks(rotation=45, ha='right'); plt.title('Confusion matrix')
plt.show()

test.to_csv('openrouter_embedding_results.csv', index=False)
print('Saved openrouter_embedding_results.csv')

## Discussion

- Which categories were confused?
- Is the error caused by the model, the gold labels, or overlapping category definitions?
- What threshold would you choose for screening versus publication?
- What metadata would make each result traceable: document ID, page, date, jurisdiction, quotation?
- How would performance change with Persian or multilingual text?

**Replication note:** exact accuracy and the PCA picture may differ slightly because of model/provider versions and the chosen threshold. Compare the workflow and the error analysis, not only a single score.